In [82]:
import duckdb
import random
from functions.analyte import ANALYTES
from networks.cnn_dilated_convolutions import CNNModel
from networks.auto_encoder import AutoencoderModel
from functions.evaluation import evaluate
from functions.full_model import predict


con = duckdb.connect('../capillary.db')

df = con.execute(""" 
                 SELECT row_id, age, gender, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set
                 FROM protein_data 
                 WHERE set = 'test'
                 AND observation_nr = 1
                 """).df()

df = predict(df)
con.close()


In [ ]:
import numpy as np

df['type'] = 'normal'

#slår ihop oligoklonalt och lätt avvikande
df.loc[df['label']==4,'label'] = 2
df.loc[df['final_prediction']==4,'final_prediction'] = 2


mat = np.zeros((3,3))
total_missclassified = sum(df['label'] != df['final_prediction'])
for i in range(3):
    for j in range(3):
        mat[i,j] = sum((df['label'] == i) & (df['final_prediction'] == j))
        #ids |= set(df.loc[(df['label'] == i) & (df['final_prediction'] == j),'row_id' ].sample(int(np.ceil(mat[i,j]))))

ids = set(df.loc[(df['label'] == 1) & (df['final_prediction'] == 0),'row_id' ]) 
ids |= set(df.loc[(df['label'] == 1) & (df['final_prediction'] == 2),'row_id' ])


ids |= set(df.loc[(df['label'] == 0) & (df['final_prediction'] == 1),'row_id' ].sample(21))
ids |= set(df.loc[(df['label'] == 0) & (df['final_prediction'] == 2),'row_id' ].sample(21))
ids |= set(df.loc[(df['label'] == 2) & (df['final_prediction'] == 0),'row_id' ].sample(21))
ids |= set(df.loc[(df['label'] == 2) & (df['final_prediction'] == 1),'row_id' ].sample(21))
ids |= set(df.loc[(df['label'] != df['final_prediction']),'row_id' ].sample(3))
df.loc[df['row_id'].isin(ids),'type'] = 'difficult'


ids |= set(df.loc[(df['label'] == 0) & (~df['row_id'].isin(ids)), 'row_id'].sample(50,random_state=42)) 
ids |= set(df.loc[(df['label'] == 1) & (~df['row_id'].isin(ids)), 'row_id'].sample(50,random_state=43))


print(df[df['type'] == 'difficult'].shape[0])

print(len(ids))
print(ids)

100
200
{53763, 141829, 139781, 143369, 53259, 64525, 48144, 87057, 184858, 63008, 177188, 90661, 144422, 146477, 2606, 119855, 148020, 2614, 181305, 6719, 98369, 128579, 152132, 103495, 57419, 16464, 84562, 66642, 109141, 119383, 168540, 137311, 47200, 73320, 97897, 33898, 156266, 83053, 124531, 115, 49781, 84086, 78456, 67705, 91771, 58494, 126593, 52354, 35462, 35465, 94346, 37515, 32911, 140944, 58004, 80021, 6294, 89748, 178329, 28317, 180383, 76961, 144033, 176809, 115884, 117423, 149688, 42169, 38074, 114362, 62143, 134848, 9923, 12996, 106693, 128712, 102091, 148684, 128205, 73422, 122572, 59600, 76500, 77525, 160980, 163035, 52957, 50918, 126185, 24811, 66802, 185079, 176899, 120584, 61711, 3347, 150804, 160019, 158486, 91415, 155926, 109845, 43802, 159007, 100640, 108324, 6445, 110382, 177970, 153398, 51513, 16699, 38212, 122694, 158024, 34635, 11597, 26958, 103248, 166224, 152913, 103251, 22875, 175451, 62813, 57182, 95586, 106854, 128360, 49514, 33645, 166254, 180079, 39793

In [76]:
df = df[df['row_id'].isin(ids)]
df = df.sample(frac=1).reset_index(drop=True)
print(len(df))


df = df[['row_id','type','value','albumin','antitrypsin','orosomukoid','haptoglobin','crp','igg','iga','igm']]
con = duckdb.connect('../application/application.db')
con.execute('DROP TABLE IF EXISTS difficult_cases')
con.execute('CREATE TABLE difficult_cases AS SELECT * FROM df')

con.close()


200
